# AI-Powered Product Description Generator

**Project:** AI in Retail and E-commerce  
**Platform:** Google Colab  
**LLM approach:** Hugging Face Transformers (Option B)  
**Model:** `gpt2`

## Project goal
Enter a product name, keywords, and a preferred tone. The language model generates a product description that can be used as a starting point for an online retail listing.

This notebook implements the assignment as a Colab notebook rather than a Flask website. It includes the optional tone-selection stretch goal.

> **Important:** GPT-2 may produce inaccurate or invented product details. Review every description before publishing it. Only provide product facts that you can verify.


## 1. Install and import libraries

Run this cell first. Colab may take a few minutes to install Transformers and download the model.

In [1]:
!pip -q install transformers torch

In [2]:
import re
import torch
from transformers import pipeline

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())


PyTorch version: 2.11.0+cpu
GPU available: False


## 2. Load the language model

The assignment suggests GPT-2 as a small starting model. The text-generation pipeline makes it easier to use without building or training a model.

The model is loaded once and reused for each description.

In [3]:
MODEL_NAME = "gpt2"

device = 0 if torch.cuda.is_available() else -1

generator = pipeline(
    task="text-generation",
    model=MODEL_NAME,
    device=device
)

print(f"Loaded {MODEL_NAME} successfully.")
print("Running on:", "GPU" if device == 0 else "CPU")


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Loaded gpt2 successfully.
Running on: CPU


## 3. Create the description-generation function

The function:
- checks that a product name and at least one keyword are provided;
- adds the selected tone to the prompt;
- asks for a concise retail description;
- removes the original prompt from the generated continuation; and
- returns a readable result.

The model generates suggested marketing text. It does not verify product specifications.

In [4]:
def generate_description(product_name, keywords, tone="neutral"):
    # Basic input validation
    product_name = str(product_name).strip()
    if not product_name:
        raise ValueError("Please enter a product name.")

    if isinstance(keywords, str):
        keyword_list = [word.strip() for word in keywords.split(",") if word.strip()]
    else:
        keyword_list = [str(word).strip() for word in keywords if str(word).strip()]

    if not keyword_list:
        raise ValueError("Please enter at least one keyword.")

    allowed_tones = {
        "neutral", "formal", "casual", "humorous",
        "for young adults", "for professionals"
    }
    if tone not in allowed_tones:
        tone = "neutral"

    keyword_text = ", ".join(keyword_list)

    prompt = (
        f"Write a clear and compelling online retail product description. "
        f"Product: {product_name}. "
        f"Keywords to include: {keyword_text}. "
        f"Writing tone: {tone}. "
        f"Do not invent technical specifications, certifications, or guarantees. "
        f"Product description:"
    )

    try:
        result = generator(
            prompt,
            max_new_tokens=90,
            num_return_sequences=1,
            do_sample=True,
            temperature=0.8,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=generator.tokenizer.eos_token_id
        )[0]["generated_text"]

        description = result[len(prompt):].strip()

        # Clean up common generation artifacts
        description = re.split(r"\n\s*\n", description)[0].strip()
        description = re.sub(r"\s+", " ", description)

        if not description:
            return "The model did not produce a description. Please try again."

        return description

    except Exception as error:
        return f"Generation error: {error}"


## 4. Try a sample product

Run this example to confirm that the model works. Change the product name, keywords, or tone to experiment.

In [5]:
sample_description = generate_description(
    product_name="Leather ankle boots",
    keywords="durable, stylish, comfortable, everyday wear",
    tone="casual"
)

print("GENERATED PRODUCT DESCRIPTION")
print("-" * 35)
print(sample_description)


GENERATED PRODUCT DESCRIPTION
-----------------------------------
A simple but effective way of carrying out business in the home on any given day by one person with no other tools than your feet is an excellent example of how it can be done easily from anywhere for anyone who wants to use them safely." Hip-hop clothing line has been taking off since 2001 when they started their own company called Hippo! The first two items (Salsam Bikinis), were made available at wholesale prices


## 5. Enter your own product information

Use the input boxes below to generate a description. Keywords should be separated by commas.

**Colab note:** This interactive form uses `ipywidgets`. If the controls do not appear, run the cell again or use the manual-input alternative in the next section.

In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

product_input = widgets.Text(
    value="",
    placeholder="Example: Stainless steel water bottle",
    description="Product:",
    layout=widgets.Layout(width="85%")
)

keywords_input = widgets.Text(
    value="",
    placeholder="Example: reusable, insulated, lightweight",
    description="Keywords:",
    layout=widgets.Layout(width="85%")
)

tone_input = widgets.Dropdown(
    options=[
        ("Neutral", "neutral"),
        ("Formal", "formal"),
        ("Casual", "casual"),
        ("Humorous", "humorous"),
        ("For young adults", "for young adults"),
        ("For professionals", "for professionals")
    ],
    value="neutral",
    description="Tone:"
)

generate_button = widgets.Button(
    description="Generate Description",
    button_style="primary",
    icon="magic"
)

output_area = widgets.Output()

def on_generate_clicked(_):
    with output_area:
        clear_output()
        try:
            description = generate_description(
                product_input.value,
                keywords_input.value,
                tone_input.value
            )
            print("GENERATED PRODUCT DESCRIPTION")
            print("-" * 35)
            print(description)
            print("\nReminder: Check every product claim for accuracy before publishing.")
        except ValueError as error:
            print("Input error:", error)

generate_button.on_click(on_generate_clicked)

display(
    product_input,
    keywords_input,
    tone_input,
    generate_button,
    output_area
)


Text(value='', description='Product:', layout=Layout(width='85%'), placeholder='Example: Stainless steel water…

Text(value='', description='Keywords:', layout=Layout(width='85%'), placeholder='Example: reusable, insulated,…

Dropdown(description='Tone:', options=(('Neutral', 'neutral'), ('Formal', 'formal'), ('Casual', 'casual'), ('H…

Button(button_style='primary', description='Generate Description', icon='magic', style=ButtonStyle())

Output()

## 6. Manual-input alternative

If the interactive controls above do not work, edit the values in this cell and run it.

In [7]:
product_name = "Wireless headphones"
keywords = "comfortable, portable, clear sound, rechargeable"
tone = "for professionals"

description = generate_description(product_name, keywords, tone)
print(description)


A set of accessories with various features that will help you avoid unnecessary clutter in your home when it comes time to use them more often than the other way around. You need an HTML5 capable browser such as Chrome , Firefox or Safari to see this content . Play Replay With rel=nofollow >html&setTimeout(function(){ var d = document.createElement('script'); if (d) { console && !document; } else


## 7. Compare different tones

This experiment uses the same product and keywords with different tones. Compare the results and consider which audience each version might appeal to.

Because GPT-2 is a general-purpose model, tone differences may be subtle or inconsistent.

In [8]:
test_product = "Reusable coffee cup"
test_keywords = "reusable, lightweight, convenient, travel"

for selected_tone in ["formal", "casual", "humorous", "for professionals"]:
    print(f"\nTONE: {selected_tone.upper()}")
    print("-" * 50)
    print(generate_description(test_product, test_keywords, selected_tone))



TONE: FORMAL
--------------------------------------------------
An innovative new way of storing information about your personal experiences with the use of portable devices (e-cigs, eReader books). The company has released its latest mobile app for Android that will allow users to track their usage by writing simple descriptions on an iPhone as well! Posted in Coffee Beans | 12 Comments

TONE: CASUAL
--------------------------------------------------
A simple white label with your name on it that will make the reader understand how you have worked out this unique design for them without any need for extra information about their needs – all of which is perfect if they are looking at an average person's products from different brands in order to determine what kind of stuff would be best suited as much like buying something similar sized but also using less expensive items so we can get more bang-for our buck here than

TONE: HUMOROUS
--------------------------------------------------

## 8. Testing and evaluation

Use the checklist below after running the notebook.

- [ ] The model loads without an error.
- [ ] A product name and keywords generate a description.
- [ ] Different tones are passed into the prompt.
- [ ] Empty product names are rejected.
- [ ] Empty keyword lists are rejected.
- [ ] The generated description is readable.
- [ ] Product claims are checked against reliable product information.

### Questions for reflection
1. Which tone produced the most useful description?
2. Did the model use the keywords naturally?
3. Did it invent any product features or promises?
4. How would you improve the prompt or model for a real online store?


## 9. Connection to AI in Retail and E-commerce

This project demonstrates a simplified personalized shopping experience. A retailer could use product information and customer-segment preferences to create different versions of product copy. For example, a professional tone may suit business customers, while a casual tone may be more appropriate for a general audience.

The prompt and keywords act as input data that influence the LLM's output. Better, more accurate product information can help produce more relevant descriptions. In a real retail system, generated descriptions should be reviewed before publication. Ethical concerns include misleading claims, biased language, and using customer data without appropriate consent. The system should not invent product benefits, safety claims, certifications, or specifications.


## 10. Short project write-up (edit to reflect your actual experience)

I chose **Option B: Hugging Face Transformers** because it allows the project to use a language model in Google Colab without requiring a paid API key. I used the GPT-2 text-generation pipeline and created a function that accepts a product name, keywords, and a selected tone. The notebook also includes input validation and sample tests.

One challenge was getting the model to produce relevant product descriptions because GPT-2 can sometimes repeat text or add details that were not provided. I addressed this by giving the model clearer instructions and limiting the generated length. In the future, the application could be improved by using a stronger instruction-following model, adding verified product specifications, supporting multiple languages, and allowing a human reviewer to approve descriptions. In a real retail application, it is important to avoid misleading claims, biased wording, and misuse of customer information.


## References
- Assignment instructions: AI-Powered Product Description Generator.
- Hugging Face Transformers documentation: https://huggingface.co/docs/transformers/
- GPT-2 model page: https://huggingface.co/openai-community/gpt2

**Submission reminder:** Save or download this notebook as an `.ipynb` file and submit it according to your course instructions. Run the cells and update the project write-up so it accurately describes your own results and challenges.
